# Torneio de estimadores em múltiplas janelas

A competição compara estimators em folds temporais expanding (Bergmeir e Benítez, 2012) de outubro de 2017 a junho de 2018. O período julho–agosto de 2018 permanece fora da seleção.


## Mesmo protocolo

Em cada fold, o pré-processamento é ajustado no treino e avaliado somente no mês de validação seguinte. As baselines obrigatórias (média do treino, mediana de lead time, baseline por estado, prevalência, prazo prometido e logística com prazo + estado) entram como linhas `baseline_*` em cada fold.

A política de desbalanceamento é única: nenhum modelo usa pesos de classe, e os classificadores são calibrados com regressão isotônica (Zadrozny e Elkan, 2002; Niculescu-Mizil e Caruana, 2005) (`CalibratedClassifierCV`, `cv=3`) ajustada apenas no treino de cada fold. A regressão logística usa `StandardScaler` (Pedregosa et al., 2011). O resumo mostra média e desvio entre folds, não um ranking do holdout final.

O treino pesado deste bloco é executado uma única vez e o resultado é salvo em `outputs/runs/`. Quando o notebook do outro idioma é executado, o mesmo resultado é carregado, desde que versão (`RUN_VERSION`), configuração, dados e código do pacote coincidam. Qualquer mudança em um desses elementos invalida o cache e provoca novo treino. O parâmetro `RUN_MODE` aceita `"auto"` (carrega quando o cache é válido e treina caso contrário), `"train"` (treina sempre e sobrescreve o cache) e `"load"` (apenas carrega, com erro se o cache não existir ou divergir). Os gráficos que seguem a tabela de resumo mostram a average precision e o MAE de cada modelo por fold, com a melhor baseline tracejada, e o ganho médio sobre essa baseline com a linha do critério P-01.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

LANG = "pt"
project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "data").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from ecom_delivery_sla import (
    build_modeling_frame,
    build_order_features,
    build_supervised_orders,
    expanding_window_folds,
    read_olist_tables,
    cached_run,
    describe_run,
    run_classification_tournament_folds,
    run_regression_tournament_folds,
    tournament_summary,
)
from ecom_delivery_sla.modeling import CATEGORICAL_FEATURES, NUMERIC_FEATURES

RUN_VERSION = "V1"
RUN_MODE = "auto"  # "auto", "train" ou "load" / "auto", "train" or "load"
CACHE_ROOT = project_root / "outputs" / "runs"
RANDOM_STATE = 42
FIRST_VAL_MONTH = "2017-10"
LAST_VAL_MONTH = "2018-06"
EMBARGO_DAYS = 30

tables = read_olist_tables(
    project_root / "data" / "raw" / "olist" / "csv",
    table_names=("orders", "items", "products", "payments", "customers", "sellers", "geolocation"),
)
supervised = build_supervised_orders(tables["orders"])
candidate_features = build_order_features(
    orders=tables["orders"],
    items=tables["items"],
    products=tables["products"],
    payments=tables["payments"],
    customers=tables["customers"],
    sellers=tables["sellers"],
    geolocation=tables["geolocation"],
)
modeling_frame = build_modeling_frame(candidate_features, supervised)
folds = expanding_window_folds(
    modeling_frame,
    timestamp_col="order_purchase_timestamp",
    first_val_month=FIRST_VAL_MONTH,
    last_val_month=LAST_VAL_MONTH,
    embargo_days=EMBARGO_DAYS,
)


def compute_tournament():
    regression_results, _ = run_regression_tournament_folds(folds, random_state=RANDOM_STATE)
    classification_results, _ = run_classification_tournament_folds(folds, random_state=RANDOM_STATE)
    return {
        "regression_results": regression_results,
        "classification_results": classification_results,
    }


tournament_config = {
    "run_version": RUN_VERSION,
    "first_val_month": FIRST_VAL_MONTH,
    "last_val_month": LAST_VAL_MONTH,
    "embargo_days": EMBARGO_DAYS,
    "random_state": RANDOM_STATE,
    "numeric_features": list(NUMERIC_FEATURES),
    "categorical_features": list(CATEGORICAL_FEATURES),
}
tournament, tournament_meta = cached_run(
    "tournament",
    RUN_VERSION,
    tournament_config,
    compute_tournament,
    frames={"modeling_frame": modeling_frame},
    mode=RUN_MODE,
    cache_root=CACHE_ROOT,
)
print(describe_run(tournament_meta, language=LANG))
regression_results = tournament["regression_results"]
classification_results = tournament["classification_results"]

display(regression_results.sort_values(["fold", "rmse"]))
display(classification_results.sort_values(["fold", "average_precision"], ascending=[True, False]))
regression_summary = tournament_summary(regression_results)
classification_summary = tournament_summary(classification_results)
display(regression_summary)
display(classification_summary)

from ecom_delivery_sla_viz import plot_gain_over_baseline, plot_metric_by_fold

best_ap_baseline = classification_summary.loc[~classification_summary["is_baseline"], "average_precision_best_baseline"].iloc[0]
best_mae_baseline = regression_summary.loc[~regression_summary["is_baseline"], "mae_best_baseline"].iloc[0]
classification_models = [m for m in classification_results["model"].unique() if not m.startswith("baseline_")]
regression_models = [m for m in regression_results["model"].unique() if not m.startswith("baseline_")]
display(
    plot_metric_by_fold(
        classification_results, "fold", "model", "average_precision",
        reference=best_ap_baseline, models=classification_models, language=LANG,
    )
)
display(plot_gain_over_baseline(classification_summary, metric="average_precision", threshold=0.05, fold_results=classification_results, language=LANG))
display(
    plot_metric_by_fold(
        regression_results, "fold", "model", "mae",
        reference=best_mae_baseline, models=regression_models, lower_is_better=True, language=LANG,
    )
)
display(plot_gain_over_baseline(regression_summary, metric="mae", threshold=0.03, fold_results=regression_results, language=LANG))

Resultados carregados de outputs/runs/tournament/V1 (gerados em 2026-11-16 12:48, 97,6 min de treino; fingerprint c98673de)


,fold,model,split,mae,rmse,r2,fit_seconds,predict_seconds
2,2017-10,ElasticNet,validation,4.045401,6.305849,0.383747,0.332874,0.003235
1,2017-10,Ridge,validation,4.052816,6.316706,0.381623,0.019707,0.001880
0,2017-10,OLS,validation,4.057193,6.320104,0.380958,0.158508,0.011541
3,2017-10,RandomForest,validation,4.116908,6.376229,0.369914,87.820826,0.199953
5,2017-10,HistGradientBoosting,validation,4.053233,6.509835,0.343232,2.386196,0.071254
...,...,...,...,...,...,...,...,...
106,2018-06,baseline_median_actual_minus_promise,validation,5.060577,6.922012,0.423272,NaN,NaN
98,2018-06,ElasticNet,validation,6.646737,8.049955,0.220003,0.808409,0.006055
97,2018-06,Ridge,validation,6.714811,8.119959,0.206378,0.058313,0.002025
96,2018-06,OLS,validation,6.717600,8.124203,0.205548,1.568745,0.016294


,fold,model,split,roc_auc,average_precision,balanced_accuracy,brier,ece_10bins,precision_at_5pct,recall_at_5pct,precision_at_10pct,recall_at_10pct,fit_seconds,predict_seconds
0,2017-10,LogisticRegression,validation,0.699597,0.087470,0.500000,0.039471,0.014030,0.111607,0.133690,0.109375,0.262032,2.350236,0.022858
6,2017-10,CatBoost,validation,0.685486,0.084222,0.500000,0.039591,0.012413,0.098214,0.117647,0.102679,0.245989,15.076193,0.211698
3,2017-10,HistGradientBoosting,validation,0.682813,0.081510,0.500000,0.039629,0.012171,0.102679,0.122995,0.102679,0.245989,1.687769,0.071140
4,2017-10,XGBoost,validation,0.640908,0.077731,0.500000,0.039698,0.011749,0.098214,0.117647,0.095982,0.229947,5.651256,0.229472
9,2017-10,baseline_logistic_promise_plus_state,validation,0.665301,0.077667,0.500000,0.039611,0.004221,0.098214,0.117647,0.093750,0.224599,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
81,2018-06,RandomForest,validation,0.755461,0.048150,0.500000,0.012953,0.031090,0.055738,0.239437,0.044262,0.380282,102.755160,0.867800
80,2018-06,LogisticRegression,validation,0.757679,0.042872,0.499751,0.018207,0.062322,0.055738,0.239437,0.039344,0.338028,14.480967,0.031104
82,2018-06,ExtraTrees,validation,0.742521,0.036099,0.500000,0.013260,0.032983,0.045902,0.197183,0.047541,0.408451,78.759192,1.001772
88,2018-06,baseline_negative_promised_lead_days,validation,0.577750,0.019549,NaN,NaN,NaN,0.036066,0.154930,0.018033,0.154930,NaN,NaN


,model,n_folds,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std,fit_seconds_mean,predict_seconds_mean,is_baseline,mae_gain_vs_baseline,mae_folds_won,mae_best_baseline,rmse_gain_vs_baseline,rmse_folds_won,rmse_best_baseline
0,ExtraTrees,9,5.953134,0.846722,9.086746,1.614968,0.108495,0.214743,219.440672,0.378886,False,-0.002634,5.0,baseline_state_median_actual_minus_promise,0.012652,5.0,baseline_state_median_actual_minus_promise
1,CatBoost,9,5.947470,1.085944,9.091165,1.804447,0.102361,0.262922,7.855869,0.012115,False,-0.001680,5.0,baseline_state_median_actual_minus_promise,0.012172,4.0,baseline_state_median_actual_minus_promise
2,HistGradientBoosting,9,6.056313,1.498375,9.167484,1.946635,0.071322,0.369571,6.514050,0.316667,False,-0.020011,6.0,baseline_state_median_actual_minus_promise,0.003879,6.0,baseline_state_median_actual_minus_promise
3,baseline_state_median_actual_minus_promise,9,5.937495,1.395744,9.203186,2.425708,0.089349,0.309171,NaN,NaN,True,NaN,NaN,,NaN,NaN,
4,ElasticNet,9,6.300179,1.085398,9.283756,1.933124,0.083982,0.202140,1.111935,0.008601,False,-0.061084,4.0,baseline_state_median_actual_minus_promise,-0.008755,4.0,baseline_state_median_actual_minus_promise
5,RandomForest,9,6.380546,1.783798,9.298089,1.927566,0.032845,0.466334,241.665489,0.296233,False,-0.074619,6.0,baseline_state_median_actual_minus_promise,-0.010312,6.0,baseline_state_median_actual_minus_promise
6,XGBoost,9,6.330275,1.453125,9.352481,1.835141,0.032739,0.370277,5.246163,0.037707,False,-0.066152,4.0,baseline_state_median_actual_minus_promise,-0.016222,4.0,baseline_state_median_actual_minus_promise
7,LightGBM,9,6.331933,1.447127,9.414514,1.972219,0.022596,0.358185,1.826333,0.100790,False,-0.066432,4.0,baseline_state_median_actual_minus_promise,-0.022962,4.0,baseline_state_median_actual_minus_promise
8,Ridge,9,6.841355,1.380036,9.833554,2.302904,-0.041140,0.346694,0.058890,0.003738,False,-0.152229,2.0,baseline_state_median_actual_minus_promise,-0.068495,2.0,baseline_state_median_actual_minus_promise
9,OLS,9,6.862683,1.386298,9.855199,2.319910,-0.045542,0.348147,0.919816,0.023472,False,-0.155821,2.0,baseline_state_median_actual_minus_promise,-0.070846,2.0,baseline_state_median_actual_minus_promise


,model,n_folds,average_precision_mean,average_precision_std,roc_auc_mean,roc_auc_std,brier_mean,brier_std,ece_10bins_mean,ece_10bins_std,fit_seconds_mean,predict_seconds_mean,is_baseline,average_precision_gain_vs_baseline,average_precision_folds_won,average_precision_best_baseline
0,CatBoost,9,0.174999,0.101000,0.707008,0.038050,0.076181,0.044793,0.057619,0.044637,23.386609,0.061755,False,0.139115,8.0,baseline_logistic_promise_plus_state
1,HistGradientBoosting,9,0.156258,0.098451,0.675659,0.051234,0.079954,0.044984,0.064149,0.058121,10.748787,0.321373,False,0.017128,7.0,baseline_logistic_promise_plus_state
2,XGBoost,9,0.154524,0.086892,0.676739,0.052624,0.077973,0.045227,0.059344,0.048404,11.419012,0.139258,False,0.005840,6.0,baseline_logistic_promise_plus_state
3,baseline_logistic_promise_plus_state,9,0.153627,0.089007,0.680947,0.061342,0.075796,0.048680,0.049702,0.042565,NaN,NaN,True,NaN,NaN,
4,LightGBM,9,0.152469,0.094095,0.673275,0.051060,0.078856,0.044940,0.062677,0.050724,5.082216,0.265077,False,-0.007537,4.0,baseline_logistic_promise_plus_state
5,LogisticRegression,9,0.151856,0.089213,0.686817,0.061914,0.080559,0.045898,0.079425,0.035549,7.809579,0.020394,False,-0.011531,4.0,baseline_logistic_promise_plus_state
6,RandomForest,9,0.150238,0.097009,0.670793,0.044232,0.077087,0.044478,0.057769,0.044516,55.814494,0.727601,False,-0.022061,3.0,baseline_logistic_promise_plus_state
7,ExtraTrees,9,0.149607,0.095750,0.676382,0.041290,0.077484,0.047676,0.057258,0.043451,46.056691,0.763884,False,-0.026169,5.0,baseline_logistic_promise_plus_state
8,baseline_prevalence,9,0.083395,0.056714,0.500000,0.000000,0.077925,0.050253,0.050500,0.044912,NaN,NaN,True,NaN,NaN,
9,baseline_negative_promised_lead_days,9,0.078068,0.046667,0.503639,0.058759,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN,NaN,


### Insight

As tabelas de resumo mostram, por modelo, média e desvio entre folds, o ganho relativo sobre a melhor baseline (`*_gain_vs_baseline`, positivo é melhor) e em quantos folds o modelo vence essa baseline (`*_folds_won`). Os valores aparecem nas tabelas acima; este texto não os reproduz para não envelhecer em relação à execução.

Para `baseline_negative_promised_lead_days`, Brier e ECE (Brier, 1950; Naeini et al., 2015) ficam vazios (NaN) porque o score não é uma probabilidade. Pelo critério aprovado (P-01 em `docs/REGISTRO-DE-DECISOES.md`), um modelo só é candidato quando, na média dos folds, ganha ≥ 5% em average precision (Davis e Goadrich, 2006) (classificação) ou ≥ 3% em MAE (regressão) sobre a melhor baseline e vence essa baseline em ≥ 75% dos folds (≥ 7 de 9). A escolha final ainda considera estabilidade, calibração, interpretabilidade, complexidade e latência. Os gráficos permitem comparar visualmente a trajetória de cada modelo entre os folds e a distância do ganho médio até o critério P-01.

## Ablação da feature de distância

A hipótese é que um pedido com vários vendedores só é entregue quando chega o item do vendedor mais distante, de modo que a distância máxima (`distance_max_km`) pode representar o gargalo melhor que a média (`distance_mean_km`), hoje usada no modelo. Três variantes são comparadas, todas partindo das features atuais e trocando apenas a distância: `mean` (referência), `max` e `max_plus_range` (distância máxima mais a amplitude `distance_range_km`, diferença entre a distância máxima e a mínima).

Em cerca de 98% dos pedidos há um único vendedor ou uma única origem, e nesses casos as três medidas coincidem; por isso o efeito esperado é pequeno na validação completa. O subgrupo de pedidos com mais de um vendedor é, portanto, reportado à parte (colunas com prefixo `multi_seller_`), onde as variantes de fato diferem. Para conter o custo, apenas dois estimadores por tarefa são reavaliados nos mesmos folds do tournament acima.

A regra de leitura é fixada antes da execução: uma variante só deve substituir `mean` se melhorar a métrica principal (MAE na regressão, average precision na classificação) na média dos folds e vencer a referência em pelo menos 75% dos folds (7 de 9). Caso contrário, `mean` é mantida por simplicidade. Os gráficos da métrica principal por variante, exibidos após as tabelas, mostram a média entre folds e o desvio de cada modelo.

In [2]:
from ecom_delivery_sla import (
    ablation_summary,
    cached_run,
    describe_run,
    distance_ablation_variants,
    run_distance_ablation,
)

ablation_models = {
    "regression": ("ElasticNet", "HistGradientBoosting"),
    "classification": ("LogisticRegression", "HistGradientBoosting"),
}


def compute_ablation():
    return {
        f"{task}_ablation": run_distance_ablation(
            folds, task, models=models, random_state=RANDOM_STATE
        )
        for task, models in ablation_models.items()
    }


ablation_config = {
    "run_version": RUN_VERSION,
    "first_val_month": FIRST_VAL_MONTH,
    "last_val_month": LAST_VAL_MONTH,
    "embargo_days": EMBARGO_DAYS,
    "random_state": RANDOM_STATE,
    "variants": distance_ablation_variants(),
    "models": {task: list(models) for task, models in ablation_models.items()},
    "categorical_features": list(CATEGORICAL_FEATURES),
}
ablation, ablation_meta = cached_run(
    "distance_ablation",
    RUN_VERSION,
    ablation_config,
    compute_ablation,
    frames={"modeling_frame": modeling_frame},
    mode=RUN_MODE,
    cache_root=CACHE_ROOT,
)
print(describe_run(ablation_meta, language=LANG))
for task in ablation_models:
    display(ablation_summary(ablation[f"{task}_ablation"], reference="mean", task=task))

from ecom_delivery_sla_viz import plot_ablation

for task in ablation_models:
    display(plot_ablation(ablation_summary(ablation[f"{task}_ablation"], reference="mean", task=task), language=LANG))

Resultados carregados de outputs/runs/distance_ablation/V1 (gerados em 2026-11-16 13:01, 12,7 min de treino; fingerprint 18accc1f)


,variant,model,n_folds,multi_seller_n_mean,multi_seller_n_folds,mae_mean,mae_std,mae_delta_vs_mean,mae_folds_won,multi_seller_mae_mean,multi_seller_mae_std,multi_seller_mae_delta_vs_mean,multi_seller_mae_folds_won
0,mean,ElasticNet,9,89.0,9,6.300179,1.085398,NaN,NaN,4.664334,1.118722,NaN,NaN
1,mean,HistGradientBoosting,9,89.0,9,6.056313,1.498375,NaN,NaN,4.464082,1.853946,NaN,NaN
2,max,ElasticNet,9,89.0,9,6.300774,1.085398,-0.000094,1.0,4.684209,1.116501,-0.004261,2.0
3,max,HistGradientBoosting,9,89.0,9,6.055120,1.476146,0.000197,5.0,4.517004,1.921493,-0.011855,2.0
4,max_plus_range,ElasticNet,9,89.0,9,6.299801,1.085478,0.000060,6.0,4.650839,1.121373,0.002893,5.0
5,max_plus_range,HistGradientBoosting,9,89.0,9,6.063650,1.493304,-0.001212,3.0,4.512154,1.873316,-0.010769,4.0


,variant,model,n_folds,multi_seller_n_mean,multi_seller_n_folds,average_precision_mean,average_precision_std,average_precision_delta_vs_mean,average_precision_folds_won,multi_seller_average_precision_mean,multi_seller_average_precision_std,multi_seller_average_precision_delta_vs_mean,multi_seller_average_precision_folds_won
0,mean,LogisticRegression,9,89.0,5,0.151856,0.089213,NaN,NaN,0.149058,0.198157,NaN,NaN
1,mean,HistGradientBoosting,9,89.0,5,0.156258,0.098451,NaN,NaN,0.155513,0.096495,NaN,NaN
2,max,LogisticRegression,9,89.0,5,0.152102,0.089103,0.001626,5.0,0.148604,0.198277,-0.003044,1.0
3,max,HistGradientBoosting,9,89.0,5,0.157362,0.094520,0.007064,5.0,0.211612,0.183561,0.360729,2.0
4,max_plus_range,LogisticRegression,9,89.0,5,0.151863,0.089530,0.000050,5.0,0.164253,0.195013,0.101939,2.0
5,max_plus_range,HistGradientBoosting,9,89.0,5,0.157844,0.095896,0.010147,6.0,0.107461,0.058194,-0.308992,1.0


### Insight da ablação

As tabelas mostram, por variante e modelo, a métrica principal (média e desvio entre folds), o delta relativo sobre `mean` (positivo é melhor) e o número de folds em que a variante supera a referência, primeiro na validação completa e depois no subgrupo multi-vendedor. Os valores aparecem nas tabelas acima; este texto não os reproduz para não envelhecer em relação à execução.

A leitura segue a regra enunciada na abertura do bloco: a troca da distância só é justificável quando o ganho médio e a consistência entre folds são atendidos simultaneamente, e um ganho restrito ao subgrupo multi-vendedor, sem efeito na validação completa, não basta por si só. Na rodada de 2026-11-12, nenhuma variante atendeu a essa regra, e a manutenção de `distance_mean_km` foi aprovada (P-03 em `docs/REGISTRO-DE-DECISOES.md`); `distance_range_km` permanece no catálogo fora do modelo. Uma nova execução deste bloco serve para confirmar essa leitura, não para alterar o catálogo sem nova aprovação. Os gráficos reproduzem, por variante e modelo, a média e o desvio entre folds já tabelados.

## Critério de decisão

Para regressão, menor MAE/RMSE e estabilidade entre janelas são desejáveis. Para classificação, average precision é especialmente relevante porque a violação é uma classe minoritária; ROC AUC (Fawcett, 2006) e calibração complementam a análise. O período jul–ago/2018 não participa desta seleção.

### Insight final

O tournament serve para descobrir quais famílias de modelo merecem diagnóstico e tuning após o fechamento dos contratos de rede, features e validação multi-janela. O ganho mínimo sobre as baselines segue o critério P-01, e os tempos de treino e predição (`fit_seconds`, `predict_seconds`) devem entrar na decisão. Nenhum resultado desta etapa será tratado como avaliação final.

## Referências

Bergmeir, C. e Benítez, J. M. (2012). On the use of cross-validation for time series predictor evaluation. *Information Sciences*, 191, 192–213. https://doi.org/10.1016/j.ins.2011.12.028

Brier, G. W. (1950). Verification of forecasts expressed in terms of probability. *Monthly Weather Review*, 78(1), 1–3.

Davis, J. e Goadrich, M. (2006). The relationship between precision-recall and ROC curves. *Proceedings of the 23rd International Conference on Machine Learning*, 233–240. https://doi.org/10.1145/1143844.1143874

Fawcett, T. (2006). An introduction to ROC analysis. *Pattern Recognition Letters*, 27(8), 861–874. https://doi.org/10.1016/j.patrec.2005.10.010

Naeini, M. P., Cooper, G. F. e Hauskrecht, M. (2015). Obtaining well calibrated probabilities using Bayesian binning. *Proceedings of the 29th AAAI Conference on Artificial Intelligence*, 2901–2907. https://doi.org/10.1609/aaai.v29i1.9602

Niculescu-Mizil, A. e Caruana, R. (2005). Predicting good probabilities with supervised learning. *Proceedings of the 22nd International Conference on Machine Learning*, 625–632. https://doi.org/10.1145/1102351.1102430

Pedregosa, F. et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, 12, 2825–2830.

Zadrozny, B. e Elkan, C. (2002). Transforming classifier scores into accurate multiclass probability estimates. *Proceedings of the 8th ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 694–699. https://doi.org/10.1145/775047.775151